# 02 — Classical baselines and the QR formulation audit

**Primary task:** establish a stable, appropriately regularized classical comparison before interpreting an apparent neural advantage.

The linear Tikhonov/TSVD solvers act on exactly the same finite interior observation vector. They use a rich product-sine basis, not the two modes used to generate the synthetic reference. Initial truth is used only after a regularization parameter has been selected from the observed-data discrepancy.

The QR sections distinguish the supplied mixed formulation from a compatible/Navier closure. The 1D P1 implementation is a diagnostic of the formulation; it does **not** certify the archived 2D/3D FreeFEM results.

**Reduced plan:** the primary linear comparison uses four observation vectors (clean plus three noise draws) and two classical methods: **eight fits**. All eight are already present in your archive. The cached driver below loads them; it does not repeat the optimizer/parameter sweep. Four additional old noise-draw fits remain preserved outside the primary cohort. The displayed formulation diagnostics are inexpensive separate checks.


In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "inverse_study").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Extract the entire study ZIP and open this notebook from its folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from inverse_study.common import *
from inverse_study import numerics as num
from inverse_study import workflows as flow
RESULTS.mkdir(exist_ok=True)
plt.rcParams.update({"font.size": 11, "figure.dpi": 110, "axes.grid": True,
                     "grid.alpha": 0.2, "savefig.bbox": "tight"})
print("Study folder:", ROOT)
from inverse_study import completion as study
from inverse_study import completion_checks as checks
REDUCED = RESULTS / "reduced_plan"
REDUCED.mkdir(parents=True, exist_ok=True)
study.verify_protocol_lock(require_reference=True)


## 1. Rich-basis inversion on actual observation locations

For basis functions $\phi_k$, form $A_{ik}=\phi_k(x_i)\exp(-\mathrm{Fo}T\lambda_k)$ and solve
$\min_c\|Ac-y\|_2^2+\alpha\|Lc\|_2^2$, or truncate its singular-value expansion.
Tikhonov chooses the largest tested $\alpha$ whose residual is at most $1.1\delta$ (including the numerical floor). TSVD chooses the smallest retained rank meeting that target. An unattained target or an optimum at the strongest grid edge is reported explicitly.

For these analytical data only, the default clean numerical floor is roundoff-scaled. Numerical or experimental data require an independently justified nonzero floor.

In [ ]:
p = problem_spec("smooth2d")
folder = freeze_observations(p, 33, .05, 3100)
_, obs, meta = load_observations(folder)
cached = flow.run_spectral_case(folder, kmax=12, profile="candidate")
fits = {}
for _, row in cached.iterrows():
    run = ROOT / row.run_folder
    with np.load(run / "solution.npz", allow_pickle=False) as z:
        modes = z["modes"].copy(); coefficients = z["coefficients"].copy()
    fits[row.method] = {**row.to_dict(), "modes":modes, "coefficients":coefficients,
        "number_of_modes":len(modes), "residual_norm":row.independent_observation_residual,
        "path":pd.read_csv(run / "parameter_path.csv").to_dict("records")}
display(pd.DataFrame([{k: fit[k] for k in ["method", "number_of_modes", "numerical_rank",
    "selected_parameter", "residual_norm", "target_residual", "selection_status", "wall_seconds"]}
    for fit in fits.values()]))
_, Xeval = tensor_grid(2, 101)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
fields = [exact_linear(p, Xeval)] + [num.evaluate_spectral(fits[m], Xeval, 0, p) for m in fits]
for ax, field, title in zip(axes, fields, ["Reference: evaluation only", "Tikhonov", "TSVD"]):
    im = ax.imshow(field.reshape(101, 101).T, origin="lower", extent=(0,1,0,1), cmap="viridis", vmin=-.1, vmax=1.1)
    ax.set_title(title); ax.set_xlabel("x"); ax.set_ylabel("y")
fig.colorbar(im, ax=axes.tolist(), shrink=.8); plt.show()
baseline_dir = REDUCED / "baseline_diagnostics"
baseline_dir.mkdir(parents=True,exist_ok=True)
print("Loaded saved parameter paths and selected coefficients; no repeated fit is needed.")
print("Post-selection initial errors:",
      {m: relative_l2(num.evaluate_spectral(f, Xeval, 0, p), exact_linear(p, Xeval)) for m,f in fits.items()})


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, (method, fit) in zip(axes, fits.items()):
    path = pd.DataFrame(fit["path"])
    ax.plot(path.parameter, path.residual_norm, "o-", ms=3)
    ax.axhline(fit["target_residual"], color="k", linestyle="--", label="Discrepancy target")
    if method == "tikhonov": ax.set_xscale("log")
    ax.set(xlabel="alpha" if method == "tikhonov" else "Retained rank", ylabel="Observation residual L2", title=method)
    ax.legend()
plt.tight_layout(); plt.show()

## 2. Reuse the declared rich-basis comparison

The primary basis remains `kmax=12` (144 coefficients), preserving your completed fits. Optional basis-sensitivity outputs from the original notebook remain available under `results/baseline_diagnostics`; they do not create additional primary statistical repetitions. A cache is reused only under the original dataset, method, basis and numerical-source identity.


In [ ]:
RUN_BASELINE_COMPLETION = True
specs = study.compact_linear_plan().drop_duplicates(
    ["case", "exposure", "obs_per_dim", "noise_level", "noise_seed"])
print("Primary classical fits:",len(specs)*2,"(one Tikhonov and one TSVD per observation vector)")
if RUN_BASELINE_COMPLETION:
    tables = []
    for _, row in specs.iterrows():
        case = problem_spec(row["case"], row["exposure"])
        data = freeze_observations(case, int(row.obs_per_dim), float(row.noise_level), int(row.noise_seed))
        tables.append(flow.run_spectral_case(data, kmax=12, profile="candidate"))
    classical = pd.concat(tables, ignore_index=True)
    display(classical[["method","noise_level","noise_seed","selection_status","initial_relative_l2"]])
    classical.to_csv(REDUCED / "linear_classical_cohort.csv",index=False)
else:
    print("Classical completion is OFF.")


## 3. What the archived QR outputs actually show

These are source-reported gamma-sweep records, not a newly validated benchmark. The old files use full-field interpolation of terminal samples. This preprocessing is additional to the observed point vector and can dominate sparse-data behavior. The original FreeFEM scripts are bundled under `legacy_sources/freefem`; they retain their historical paths for provenance and are not launched automatically.

In [ ]:
qr_files = sorted((DATA / "legacy_qr").glob("qr_*.csv"))
legacy_qr = pd.concat([pd.read_csv(f).assign(source=f.name) for f in qr_files], ignore_index=True)
columns = [c for c in ["run_id", "gamma", "data_discrepancy_noisy_interior", "initial_rel_l2_validation", "source"] if c in legacy_qr]
display(legacy_qr[columns])
legacy_qr.to_csv(baseline_dir / "legacy_QR_parameter_records.csv", index=False)
print("Do not select gamma from initial_rel_l2_validation in the primary comparison.")

## 4. Executable 1D mixed-P1 closure test

The supplied weak form fixes $w$ at the boundary but permits the auxiliary test function at boundary nodes. Its natural closure differs from the sine-compatible case $w=q=0$. The test below implements both choices with the same P1 mass/stiffness matrices, gamma and time step, and then independently propagates each reconstructed initial profile.

The two curves are not interchangeable implementations of the same inverse problem. A passing algebraic solver test is not enough to accept either closure as the appropriate physical regularization.

In [ ]:
from inverse_study.qr_audit import solve_qr_1d, select_qr_by_discrepancy, run_qr_self_checks
qr_checks = run_qr_self_checks()
display(pd.Series(qr_checks).to_frame("software_check"))
write_json(REDUCED / "QR_1d_self_checks.json", qr_checks)

qproblem = {"name": "closure_diagnostic", "dim": 1, "bounds": [[0,1]], "Fo": .1, "T": 1., "law": "constant"}
qx = np.linspace(0, 1, 33)[1:-1]
qobs = {"X_obs": qx[:,None], "y_obs": np.sin(np.pi*qx)*np.exp(-.1*np.pi**2), "noise_norm": 0.0}
qr_results = []
for closure in ["archived", "navier"]:
    for gamma in [1e-5, 1e-4, 1e-3]:
        qr_results.append(solve_qr_1d(qproblem, qobs, gamma, nx=81, dt=.005, closure=closure))
display(pd.DataFrame([{k:r[k] for k in ["closure", "gamma", "stable", "residual_norm", "sensor_to_mesh_backprojection_residual", "wall_seconds"]} for r in qr_results]))
fig, ax = plt.subplots(figsize=(7,4))
for r in qr_results:
    if r["stable"] and r["gamma"] == 1e-3:
        ax.plot(r["x"], r["theta0"], label=r["closure"], linestyle="-" if r["closure"]=="archived" else "-.")
ax.plot(np.linspace(0,1,201), np.sin(np.pi*np.linspace(0,1,201)), "k--", label="Known reference: evaluation only")
ax.set(xlabel="x", ylabel="Initial temperature", title="Different auxiliary closures at gamma = 0.001")
ax.legend(); plt.show()

## 5. Decision before drafting a ranking

- Retain QR as a headline comparator only after its closure, preprocessing, gamma rule and mesh/time convergence are defensible for each reported case.
- If it fails those checks, use the validated finite-observation spectral baselines for the linear cases and a verified nonlinear comparator for the nonlinear cases. State the method change in the manuscript; do not silently relabel the old curves.
- A strongest-grid-edge selection needs an expanded parameter grid. An unattainable discrepancy needs diagnosis of discretization, representation, interpolation or model mismatch; it is not a successful selection.
- Confirm time-to-solution scopes: fitting and parameter search, observation preprocessing and forward verification should be recorded separately. The old PINN-training versus QR-inversion-plus-forward timings are not a matched speed comparison.